In [1]:
import json
import os
import re

from num2words import num2words

In [2]:
_PUNCT_PATTERN = re.compile(r"[^\w\s]")
_WHITESPACE_PATTERN = re.compile(r"\s+")
_NUMBER_PATTERN = re.compile(r"\d+")


def remove_punctuation(text: str) -> str:
    return _PUNCT_PATTERN.sub(" ", text)


def normalize_numbers(text: str, lang: str = "id") -> str:
    return _NUMBER_PATTERN.sub(lambda m: num2words(int(m.group()), lang=lang), text)


def normalize(text: str) -> str:
    text = remove_punctuation(text)
    text = normalize_numbers(text)
    text = _WHITESPACE_PATTERN.sub(" ", text).strip().lower()
    return text

In [3]:
DATA_PATH = os.path.join("..", "data", "raw", "intent_dataset.jsonl")

records = []
with open(DATA_PATH, encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print(f"loaded {len(records)} records from {DATA_PATH}")

loaded 2000 records from ../data/raw/intent_dataset.jsonl


In [4]:
for r in records:
    r["text_normalized"] = normalize(r["text"])

changed = [r for r in records if r["text"].strip().lower() != r["text_normalized"]]
print(f"{len(changed)} of {len(records)} rows changed by normalization\n")

for r in changed[:10]:
    print("RAW: ", r["text"])
    print("NORM:", r["text_normalized"])
    print()

1264 of 2000 rows changed by normalization

RAW:  eh bukan, bukan cumi goreng tepung, saya maunya nasi uduk satu ni- nih
NORM: eh bukan bukan cumi goreng tepung saya maunya nasi uduk satu ni nih

RAW:  permisi perkedel nya jadi lima aja, tadi eh saya bilang empat ya kak pak
NORM: permisi perkedel nya jadi lima aja tadi eh saya bilang empat ya kak pak

RAW:  tolong gulai ayam nya jadi empat eh lima aja, tadi saya bilang satu dong bro
NORM: tolong gulai ayam nya jadi empat eh lima aja tadi saya bilang satu dong bro

RAW:  mohon ra- rawon nya jangan jadi, batalin satu aja, sisanya tetap kak
NORM: mohon ra rawon nya jangan jadi batalin satu aja sisanya tetap kak

RAW:  btw ada menu sop buntut ga, eh es kelapa muda sama susu coklat nya berapaan harganya dong
NORM: btw ada menu sop buntut ga eh es kelapa muda sama susu coklat nya berapaan harganya dong

RAW:  mohon bukan, bukan nasi mm goreng gila, saya maunya sate ayam tiga bang
NORM: mohon bukan bukan nasi mm goreng gila saya maunya sate a

In [5]:
OUTPUT_PATH = os.path.join(
    "..", "data", "normalized", "intent_dataset_normalized.jsonl"
)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"wrote {len(records)} normalized records to {OUTPUT_PATH}")

wrote 2000 normalized records to ../data/normalized/intent_dataset_normalized.jsonl
